In [ ]:
import numpy as np

# Similarity Score
def similarity_score(gradients, reg_lambda):
    # Formula: (Sum of Gradients)^2 / (Number of samples + Lambda)
    return (np.sum(gradients)**2) / (len(gradients) + reg_lambda)

# Gain
def calculate_gain(left_grads, right_grads, parent_score, reg_lambda):
    left_score = similarity_score(left_grads, reg_lambda)
    right_score = similarity_score(right_grads, reg_lambda)
    return left_score + right_score - parent_score

def leaf_weight(gradients, reg_lambda):
    # Formula: -Sum(Gradients) / (Number of samples + Lambda)
    return -np.sum(gradients) / (len(gradients) + reg_lambda)

In [ ]:
def find_best_split(X, gradients, reg_lambda):
    best_gain = 0
    split_info = None
    parent_score = similarity_score(gradients, reg_lambda)
    
    # Check every feature
    for feature_idx in range(X.shape[1]):
        thresholds = np.unique(X[:, feature_idx])
        
        for t in thresholds:
            left_mask = X[:, feature_idx] <= t
            right_mask = X[:, feature_idx] > t
            
            if not any(left_mask) or not any(right_mask): continue
            
            gain = calculate_gain(gradients[left_mask], gradients[right_mask], parent_score, reg_lambda)
            
            if gain > best_gain:
                best_gain = gain
                split_info = {'feature': feature_idx, 'threshold': t, 'left_mask': left_mask, 'right_mask': right_mask}
                
    return split_info

In [ ]:
# Setup Data 
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1.5, 2.1, 3.8, 4.2, 5.5])

learning_rate = 0.5
reg_lambda = 1
n_iterations = 3

# Initial Prediction 
current_preds = np.full(y.shape, 0.5)
print(f"Initial Predictions: {current_preds}")

for i in range(n_iterations):
    # For MSE, Gradient = (Prediction - Actual)
    gradients = current_preds - y
    
    # Find the best split for this error
    split = find_best_split(X, gradients, reg_lambda)
    
    if split:
        # Calculate weights for the new "Left" and "Right" leaves
        val_left = leaf_weight(gradients[split['left_mask']], reg_lambda)
        val_right = leaf_weight(gradients[split['right_mask']], reg_lambda)
        
        # Update our Predictions
        # New Pred = Old Pred + (Learning Rate * Leaf Weight)
        current_preds[split['left_mask']] += learning_rate * val_left
        current_preds[split['right_mask']] += learning_rate * val_right
        
    print(f"Iteration {i+1} Errors (MSE): {np.mean((y - current_preds)**2):.4f}")

print(f"\nFinal Predictions: {current_preds}")
print(f"Actual Values: {y}")

Initial Predictions: [0.5 0.5 0.5 0.5 0.5]
Iteration 1 Errors (MSE): 4.4024
Iteration 2 Errors (MSE): 1.9708
Iteration 3 Errors (MSE): 0.9662

Final Predictions: [1.19583333 1.92083333 3.47916667 3.47916667 3.47916667]
Actual Values: [1.5 2.1 3.8 4.2 5.5]
